# Cat vs Dog Image Classification

Binary classification using a classical KNN baseline and VGG16 transfer learning.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

DATA_DIR = 'data/'
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train_images')
TEST_IMG_DIR = os.path.join(DATA_DIR, 'test_images')
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 3

## 1. Data Overview

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
train_df['filename'] = train_df['id'].astype(str) + '.jpg'

plt.figure(figsize=(5, 3))
sns.countplot(data=train_df, x='label', palette='muted')
plt.title('Cats vs Dogs')
plt.tight_layout()
plt.savefig('cv_dist_v3.png', dpi=150)
plt.show()

## 2. Split

In [ ]:
train_data, val_data = train_test_split(train_df, test_size=0.3, random_state=100, stratify=train_df['label'])
print(f"Train: {len(train_data)} | Val: {len(val_data)}")

## 3. Baseline: KNN on Colour Histograms
HSV histograms (32 bins/channel) with **K-Nearest Neighbours** (k=7, distance weighting).

In [ ]:
SIZE = (64, 64)

def color_hist(path):
    img = cv2.imread(path)
    img = cv2.resize(img, SIZE)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    f = []
    for c in range(3):
        h = cv2.calcHist([hsv], [c], None, [32], [0, 256])
        h = cv2.normalize(h, h).flatten()
        f.extend(h)
    return np.array(f)

print("Extracting colour histograms...")
X_tr = np.array([color_hist(os.path.join(TRAIN_IMG_DIR, f)) for f in train_data['filename'].values])
X_va = np.array([color_hist(os.path.join(TRAIN_IMG_DIR, f)) for f in val_data['filename'].values])
lm = {'cat': 0, 'dog': 1}
y_tr = train_data['label'].map(lm).values
y_va = val_data['label'].map(lm).values

knn = KNeighborsClassifier(n_neighbors=7, weights='distance')
knn.fit(X_tr, y_tr)
knn_preds = knn.predict(X_va)
knn_acc = accuracy_score(y_va, knn_preds)
print(f"KNN Accuracy: {knn_acc:.4f}")
print(classification_report(y_va, knn_preds, target_names=['cat', 'dog']))

## 4. VGG16 Transfer Learning
Pretrained VGG16, frozen base, 224x224 input. Augmentation: brightness + horizontal flip.

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

train_gen = ImageDataGenerator(
    rescale=1./255,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True
)
val_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    train_data, directory=TRAIN_IMG_DIR, x_col='filename', y_col='label',
    target_size=IMG_SIZE, class_mode='binary', batch_size=BATCH_SIZE, seed=100
)
val_flow = val_gen.flow_from_dataframe(
    val_data, directory=TRAIN_IMG_DIR, x_col='filename', y_col='label',
    target_size=IMG_SIZE, class_mode='binary', batch_size=BATCH_SIZE, shuffle=False
)

base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False
x = GlobalAveragePooling2D()(base.output)
x = Dropout(0.4)(x)
out = Dense(1, activation='sigmoid')(x)
model = Model(inputs=base.input, outputs=out)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
print("Training VGG16 head...")
model.fit(
    train_flow, epochs=EPOCHS, validation_data=val_flow,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)

## 5. Validation

In [ ]:
val_flow.reset()
preds = model.predict(val_flow)
pred_cls = (preds > 0.5).astype(int).flatten()
true_cls = val_flow.classes
labels = list(val_flow.class_indices.keys())

vgg_acc = accuracy_score(true_cls, pred_cls)
print(f"VGG16 Accuracy: {vgg_acc:.4f}")
print(classification_report(true_cls, pred_cls, target_names=labels))

cm = confusion_matrix(true_cls, pred_cls)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix - VGG16')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('cv_cm_v3.png', dpi=150)
plt.show()

In [ ]:
print(f"\nKNN: {knn_acc:.4f} | VGG16: {vgg_acc:.4f}")

## 6. Test Predictions

In [ ]:
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
test_df['filename'] = test_df['id'].astype(str) + '.jpg'
test_flow = val_gen.flow_from_dataframe(
    test_df, directory=TEST_IMG_DIR, x_col='filename', y_col=None,
    target_size=IMG_SIZE, class_mode=None, batch_size=BATCH_SIZE, shuffle=False
)

test_preds = model.predict(test_flow)
idx = val_flow.class_indices
rev = {v: k for k, v in idx.items()}
test_df['prediction'] = [rev[1] if p > 0.5 else rev[0] for p in test_preds]
test_df[['id', 'prediction']].to_csv('test-predictions_v3.csv', index=False)
print(f"Saved {len(test_df)} predictions")
test_df[['id', 'prediction']].head()